### 데이터 로딩 + 확인

In [ ]:
# 데이터 로딩

import pandas as pd

# df = pd.read_csv('Chemical_Numeric_Data_Quality.csv', encoding='cp949')

df = pd.read_csv('/content/drive/MyDrive/_for_exercise/Chemical_Numeric_Data_Quality.csv', encoding='cp949')


In [ ]:
# 가독성 향상을 위해 소수점 출력 포맷을 소수점 넷째자리까지 나오도록 설정
pd.set_option('display.float_format', lambda x: '%.4f' % x)

In [ ]:
# column명 정리
df.columns = ['Lot', 'Temp', 'Viscosity', 'Failure', 'FailureCode']


In [ ]:
# FailureCode 값 처리
df.FailureCode = df.FailureCode.fillna('None').astype(str)
df.FailureCode = df.FailureCode.apply(lambda x: x if x=='None' else str(int(float(x))))

In [ ]:
df.sample(5)

In [ ]:
df.info()

In [ ]:
df.describe()

### STEP ① 완전성 지표

In [ ]:
# 결측치 확인
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
# 결측치 처리
# 평균값으로 대체가 적절하지 않다.
# 오른쪽(큰 값)으로 치우친 분포를 보여서 평균값 자체가 이상치 영역에 속해버림.
# temp_mean = df.Temp.mean()
# df.Temp = df.Temp.fillna(temp_mean)

# visco_mean = df.Viscosity.mean()
# df.Viscosity = df.Viscosity.fillna(visco_mean)

# 중앙값으로 대체하자.
temp_median = df.Temp.median()
df.Temp = df.Temp.fillna(temp_median)

visco_median = df.Viscosity.median()
df.Viscosity = df.Viscosity.fillna(visco_median)

In [ ]:
df.info()

In [ ]:
df.describe()

### STEP ③ 일관성 지표

In [ ]:
df.Failure.value_counts()

In [ ]:
df.Failure = df.Failure.map({'정상': 0, '0': 0,
                '불량': 1, '1': 1})

df.info()

In [ ]:
df.Failure.value_counts()

### 이상치 탐색: 데이터 분포 확인

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# 데이터 분포 확인
plt.figure(figsize=(8,6))
sns.histplot(data=df, x='Temp', hue='Failure', multiple='stack', binwidth=10) #, kde=True
plt.show()

In [ ]:
# 이상치 확인
plt.figure(figsize=(8,6))

#sns.boxplot(data=df, x='Temp')
sns.boxplot(data=df, x='Failure', y='Temp')
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.histplot(data=df, x='Viscosity', hue='Failure', multiple='stack', binwidth=2) #, kde=True
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
#sns.boxplot(data=df, x='Viscosity')
sns.boxplot(data=df, x='Failure', y='Viscosity')
plt.show()

### STEP ④ 유일성 지표

In [ ]:
df.info()

In [ ]:
# 중복값이 있는지 확인
df.Lot.duplicated().sum()

In [ ]:
# 중복값 삭제
df = df.drop_duplicates(subset=['Lot'], keep='first')

In [ ]:
df.info()

### STEP ⑤ 정확성 지표

In [ ]:
df.sample(5)

In [ ]:
df.info()

In [ ]:
# CASE1 : Failure 값이 정상인데 Failure code에 값이 있는 경우
#   기존 처리 방식 : Failure code를 None으로 변경

#   더 나을 것 같은 처리 : Temp와 Viscosity가 이상치 범주라면 Failure를 '불량'으로,
#                        : Temp와 Viscosity가 정상 범주라면, Failure code를 'None'으로
#                   ※ 이상치를 처리한 상태라면, 할 수 없는 처리임.

In [ ]:
# CASE1에 속하는지 체크
incorrect_case1 = (df.Failure == 0) & (df.FailureCode != 'None')
incorrect_case1.value_counts()

In [ ]:
# 유효성 관리 범위 체크
outlier_checker = (df.Temp < 20) | (df.Temp > 30) | (df.Viscosity > 3)

# CASE1 중에서 Temp와 Viscosity가 이상치 범주라면 Failure를 '불량'으로,
df.loc[incorrect_case1 & outlier_checker, 'Failure'] = 1

# CASE1 중에서 Temp와 Viscosity가 정상 범주라면, Failure code를 'None'으로
df.loc[incorrect_case1 & ~outlier_checker, 'FailureCode'] = 'None'

In [ ]:
# 처리 결과 확인
incorrect_case1 = (df.Failure == 0) & (df.FailureCode != 'None')
incorrect_case1.value_counts()

In [ ]:
# CASE2 : Failure 값이 불량인데 Failure code에 값이 없는 경우
#   기존 처리 방식 : 데이터 삭제

#   시도해 볼 방식 : Failure code의 최빈값으로 채우기 (None을 제외한 최빈값)


In [ ]:
# CASE2에 속하는지 체크
incorrect_case2 = (df.Failure == 1) & (df.FailureCode == 'None')
incorrect_case2.value_counts()


In [ ]:
# None을 제외한 최빈값
failuerCode_mode = df[df.FailureCode != 'None'].FailureCode.mode()


In [ ]:
df.loc[incorrect_case2, 'FailureCode'] = failuerCode_mode[0]

In [ ]:
# 처리 결과 확인
incorrect_case2 = (df.Failure == 1) & (df.FailureCode == 'None')
incorrect_case2.value_counts()

### 모든 처리가 끝난 데이터

In [ ]:
df.info()

In [ ]:
df.sample(10)

In [ ]:
df.describe()